# 04 - Modelos de Regresión para Predicción de Precios de Vivienda

En este notebook implementaremos y compararemos diferentes modelos de regresión lineal para predecir precios de vivienda.

## Modelos a implementar:
- **Regresión Lineal Simple**: Una variable predictora
- **Regresión Lineal Múltiple**: Múltiples variables predictoras
- **Ridge Regression**: Regularización L2
- **Lasso Regression**: Regularización L1
- **Elastic Net**: Combinación de Ridge y Lasso

## Objetivos:
1. Implementar modelos de regresión lineal
2. Aplicar regularización para prevenir overfitting
3. Optimizar hiperparámetros
4. Comparar el rendimiento de los modelos

In [ ]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Configurar estilo de visualización
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## 1. Cargar datos procesados

In [ ]:
# Cargar datos de entrenamiento y prueba
X_train = pd.read_csv('../data/processed/X_train_scaled.csv')
X_test = pd.read_csv('../data/processed/X_test_scaled.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').iloc[:, 0]
y_test = pd.read_csv('../data/processed/y_test.csv').iloc[:, 0]

print(f"Datos de entrenamiento: {X_train.shape}")
print(f"Datos de prueba: {X_test.shape}")
print(f"Características disponibles: {list(X_train.columns)}")

## 2. Función de evaluación de modelos

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name):
    """
    Evalúa un modelo de regresión y retorna métricas
    """
    # Entrenar modelo
    model.fit(X_train, y_train)
    
    # Predicciones
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Métricas
    train_mse = mean_squared_error(y_train, y_train_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)
    train_rmse = np.sqrt(train_mse)
    test_rmse = np.sqrt(test_mse)
    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    
    # Cross-validation
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
    cv_r2_mean = cv_scores.mean()
    cv_r2_std = cv_scores.std()
    
    # Crear dataframe de resultados
    results = {
        'Model': model_name,
        'Train_RMSE': train_rmse,
        'Test_RMSE': test_rmse,
        'Train_MAE': train_mae,
        'Test_MAE': test_mae,
        'Train_R2': train_r2,
        'Test_R2': test_r2,
        'CV_R2_Mean': cv_r2_mean,
        'CV_R2_Std': cv_r2_std,
        'Overfitting': train_r2 - test_r2
    }
    
    return results, y_train_pred, y_test_pred, model

## 3. Regresión Lineal Simple

Comenzamos con regresión lineal simple usando la característica más importante identificada en el EDA: RM (número promedio de habitaciones).

In [ ]:
# Regresión lineal simple con RM (número de habitaciones)
X_train_simple = X_train[['RM']]
X_test_simple = X_test[['RM']]

# Crear y entrenar modelo
lr_simple = LinearRegression()
lr_simple_results, lr_simple_train_pred, lr_simple_test_pred, lr_simple_model = evaluate_model(
    lr_simple, X_train_simple, X_test_simple, y_train, y_test, 'Linear Regression (Simple)'
)

print(f"Regresión Lineal Simple - Test R²: {lr_simple_results['Test_R2']:.4f}")
print(f"Regresión Lineal Simple - Test RMSE: {lr_simple_results['Test_RMSE']:.2f}")
print(f"Coeficiente RM: {lr_simple_model.coef_[0]:.4f}")
print(f"Intercepto: {lr_simple_model.intercept_:.4f}")

In [ ]:
# Visualizar regresión lineal simple
plt.figure(figsize=(10, 6))
plt.scatter(X_test_simple['RM'], y_test, alpha=0.5, label='Datos reales')
plt.plot(X_test_simple['RM'], lr_simple_test_pred, color='red', linewidth=2, label='Predicción')
plt.xlabel('Número de habitaciones (RM)')
plt.ylabel('Precio (MEDV)')
plt.title('Regresión Lineal Simple: RM vs Precio')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. Regresión Lineal Múltiple

Ahora usamos todas las características disponibles para construir un modelo de regresión lineal múltiple.

In [ ]:
# Regresión lineal múltiple con todas las características
lr_multiple = LinearRegression()
lr_multiple_results, lr_multiple_train_pred, lr_multiple_test_pred, lr_multiple_model = evaluate_model(
    lr_multiple, X_train, X_test, y_train, y_test, 'Linear Regression (Multiple)'
)

print(f"Regresión Lineal Múltiple - Test R²: {lr_multiple_results['Test_R2']:.4f}")
print(f"Regresión Lineal Múltiple - Test RMSE: {lr_multiple_results['Test_RMSE']:.2f}")

# Mostrar coeficientes de las características
feature_coef = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': lr_multiple_model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print("\nTop 10 coeficientes más importantes:")
print(feature_coef.head(10))

## 5. Ridge Regression (Regularización L2)

Ridge regression añade una penalización L2 a los coeficientes, lo que ayuda a prevenir overfitting cuando hay multicolinealidad entre variables.

In [ ]:
# Definir hiperparámetros para Ridge
ridge_params = {
    'alpha': [0.1, 1.0, 10.0, 100.0, 1000.0]
}

# Grid Search para Ridge
print("Optimizando Ridge Regression...")
ridge_model = Ridge()
ridge_grid = GridSearchCV(ridge_model, ridge_params, cv=5, scoring='r2')
ridge_grid.fit(X_train, y_train)

print(f"Mejor alpha: {ridge_grid.best_params_['alpha']}")
print(f"Mejor score CV: {ridge_grid.best_score_:.4f}")

# Evaluar mejor modelo Ridge
ridge_results, ridge_train_pred, ridge_test_pred, ridge_best_model = evaluate_model(
    ridge_grid.best_estimator_, X_train, X_test, y_train, y_test, 'Ridge Regression'
)

print(f"Ridge Regression - Test R²: {ridge_results['Test_R2']:.4f}")
print(f"Ridge Regression - Test RMSE: {ridge_results['Test_RMSE']:.2f}")

## 6. Lasso Regression (Regularización L1)

Lasso regression añade una penalización L1 que puede llevar coeficientes exactamente a cero, realizando selección de características automática.

In [ ]:
# Definir hiperparámetros para Lasso
lasso_params = {
    'alpha': [0.001, 0.01, 0.1, 1.0, 10.0]
}

# Grid Search para Lasso
print("Optimizando Lasso Regression...")
lasso_model = Lasso(max_iter=10000)
lasso_grid = GridSearchCV(lasso_model, lasso_params, cv=5, scoring='r2')
lasso_grid.fit(X_train, y_train)

print(f"Mejor alpha: {lasso_grid.best_params_['alpha']}")
print(f"Mejor score CV: {lasso_grid.best_score_:.4f}")

# Evaluar mejor modelo Lasso
lasso_results, lasso_train_pred, lasso_test_pred, lasso_best_model = evaluate_model(
    lasso_grid.best_estimator_, X_train, X_test, y_train, y_test, 'Lasso Regression'
)

print(f"Lasso Regression - Test R²: {lasso_results['Test_R2']:.4f}")
print(f"Lasso Regression - Test RMSE: {lasso_results['Test_RMSE']:.2f}")

# Características seleccionadas por Lasso
selected_features = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient': lasso_best_model.coef_
})

selected_features = selected_features[selected_features['Coefficient'] != 0].sort_values('Coefficient', key=abs, ascending=False)
print(f"\nCaracterísticas seleccionadas por Lasso ({len(selected_features)}):")
print(selected_features)

## 7. Elastic Net Regression

Elastic Net combina las penalizaciones L1 y L2 de Lasso y Ridge, proporcionando un balance entre selección de características y regularización.

In [ ]:
# Definir hiperparámetros para Elastic Net
elastic_params = {
    'alpha': [0.001, 0.01, 0.1, 1.0],
    'l1_ratio': [0.1, 0.5, 0.7, 0.9]
}

# Grid Search para Elastic Net
print("Optimizando Elastic Net...")
elastic_model = ElasticNet(max_iter=10000)
elastic_grid = GridSearchCV(elastic_model, elastic_params, cv=5, scoring='r2')
elastic_grid.fit(X_train, y_train)

print(f"Mejores hiperparámetros: {elastic_grid.best_params_}")
print(f"Mejor score CV: {elastic_grid.best_score_:.4f}")

# Evaluar mejor modelo Elastic Net
elastic_results, elastic_train_pred, elastic_test_pred, elastic_best_model = evaluate_model(
    elastic_grid.best_estimator_, X_train, X_test, y_train, y_test, 'Elastic Net'
)

print(f"Elastic Net - Test R²: {elastic_results['Test_R2']:.4f}")
print(f"Elastic Net - Test RMSE: {elastic_results['Test_RMSE']:.2f}")

## 8. Comparación de Modelos Lineales

In [ ]:
# Consolidar todos los resultados
all_results = pd.DataFrame([
    lr_simple_results,
    lr_multiple_results,
    ridge_results,
    lasso_results,
    elastic_results
])

# Mostrar tabla de resultados
print("Comparación de Modelos de Regresión Lineal:")
print("="*80)
display(all_results.round(4))

# Identificar mejor modelo
best_model_idx = all_results['Test_R2'].idxmax()
best_model_name = all_results.loc[best_model_idx, 'Model']
print(f"\nMejor modelo de regresión lineal: {best_model_name}")
print(f"R² Score: {all_results.loc[best_model_idx, 'Test_R2']:.4f}")
print(f"RMSE: {all_results.loc[best_model_idx, 'Test_RMSE']:.2f}")
print(f"Overfitting: {all_results.loc[best_model_idx, 'Overfitting']:.4f}")

In [ ]:
# Visualizar comparación de métricas
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# R² Score
axes[0,0].bar(all_results['Model'], all_results['Test_R2'], color='skyblue')
axes[0,0].set_title('R² Score por Modelo')
axes[0,0].set_ylabel('R² Score')
axes[0,0].tick_params(axis='x', rotation=45)

# RMSE
axes[0,1].bar(all_results['Model'], all_results['Test_RMSE'], color='lightcoral')
axes[0,1].set_title('RMSE por Modelo')
axes[0,1].set_ylabel('RMSE')
axes[0,1].tick_params(axis='x', rotation=45)

# MAE
axes[1,0].bar(all_results['Model'], all_results['Test_MAE'], color='lightgreen')
axes[1,0].set_title('MAE por Modelo')
axes[1,0].set_ylabel('MAE')
axes[1,0].tick_params(axis='x', rotation=45)

# Overfitting
axes[1,1].bar(all_results['Model'], all_results['Overfitting'], color='gold')
axes[1,1].set_title('Overfitting (Train R² - Test R²)')
axes[1,1].set_ylabel('Diferencia R²')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 9. Análisis de Residuos del Mejor Modelo

In [ ]:
# Seleccionar el mejor modelo para análisis detallado
if best_model_name == 'Linear Regression (Simple)':
    best_model = lr_simple_model
    best_test_pred = lr_simple_test_pred
elif best_model_name == 'Linear Regression (Multiple)':
    best_model = lr_multiple_model
    best_test_pred = lr_multiple_test_pred
elif best_model_name == 'Ridge Regression':
    best_model = ridge_best_model
    best_test_pred = ridge_test_pred
elif best_model_name == 'Lasso Regression':
    best_model = lasso_best_model
    best_test_pred = lasso_test_pred
else:
    best_model = elastic_best_model
    best_test_pred = elastic_test_pred

# Calcular residuos
residuos = y_test - best_test_pred

# Visualizar residuos
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Residuos vs Predichos
axes[0,0].scatter(best_test_pred, residuos, alpha=0.6)
axes[0,0].axhline(y=0, color='r', linestyle='--')
axes[0,0].set_xlabel('Valores Predichos')
axes[0,0].set_ylabel('Residuos')
axes[0,0].set_title(f'Residuos vs Predichos - {best_model_name}')

# Histograma de residuos
axes[0,1].hist(residuos, bins=30, edgecolor='black', alpha=0.7)
axes[0,1].set_xlabel('Residuos')
axes[0,1].set_ylabel('Frecuencia')
axes[0,1].set_title(f'Histograma de Residuos - {best_model_name}')

# Q-Q plot
from scipy import stats
stats.probplot(residuos, dist="norm", plot=axes[1,0])
axes[1,0].set_title(f'Q-Q Plot - {best_model_name}')

# Valores reales vs predichos
axes[1,1].scatter(y_test, best_test_pred, alpha=0.6)
axes[1,1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1,1].set_xlabel('Valores Reales')
axes[1,1].set_ylabel('Valores Predichos')
axes[1,1].set_title(f'Reales vs Predichos - {best_model_name}')

plt.tight_layout()
plt.show()

# Estadísticas de residuos
print(f"Estadísticas de residuos para {best_model_name}:")
print(f"Media: {residuos.mean():.4f}")
print(f"Desviación estándar: {residuos.std():.4f}")
print(f"MSE de residuos: {np.mean(residuos**2):.4f}")

## 10. Guardar Modelos y Resultados

In [ ]:
import joblib
import os

# Crear directorio para modelos si no existe
os.makedirs('../models', exist_ok=True)

# Guardar modelos de regresión lineal
joblib.dump(lr_simple_model, '../models/linear_regression_simple.pkl')
joblib.dump(lr_multiple_model, '../models/linear_regression_multiple.pkl')
joblib.dump(ridge_best_model, '../models/ridge_regression.pkl')
joblib.dump(lasso_best_model, '../models/lasso_regression.pkl')
joblib.dump(elastic_best_model, '../models/elastic_net.pkl')

# Guardar resultados
all_results.to_csv('../reports/linear_models_results.csv', index=False)

print("Modelos de regresión lineal guardados exitosamente!")
print(f"Mejor modelo: {best_model_name}")
print(f"R² Score: {all_results.loc[best_model_idx, 'Test_R2']:.4f}")
print(f"RMSE: {all_results.loc[best_model_idx, 'Test_RMSE']:.2f}")
print(f"Overfitting controlado: {all_results.loc[best_model_idx, 'Overfitting']:.4f}")

## 11. Conclusiones y Recomendaciones

### Resultados Principales:

1. **Mejor Modelo Lineal**: {} con R² = {:.4f}
2. **Mejora vs Simple**: {:.2f}% de mejora respecto a regresión lineal simple
3. **Regularización efectiva**: {} mostró mejor control de overfitting
4. **Selección de características**: {} seleccionó {} características relevantes

### Observaciones:
- La regresión múltiple mejoró significativamente vs la simple
- La regularización ayudó a controlar el overfitting
- {} fue la característica más importante en todos los modelos
- Los residuos muestran {}

### Próximos Pasos:
1. En el notebook 05, implementaremos modelos avanzados (Random Forest, XGBoost)
2. Compararemos todos los modelos en el notebook 06
3. Seleccionaremos el mejor modelo general para despliegue
4. Crearemos un pipeline completo de predicción